# Table Linkage Statistics


In [ ]:
# === notebook config (auto-managed; edit values, not the tag) ===
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Database
DB_PATH = "../data/humans_clean.duckdb"

# Figure style — minimal, Nature/Science publication standard
FIGSIZE = (8, 5)
DPI = 120
FONT_TITLE = 16
FONT_LABEL = 13
FONT_TICK = 11
FONT_LEGEND = 10

# Light, restrained palette (avoid AI-slop saturation)
COLOR_PRIMARY = "#2171b5"
COLOR_SECONDARY = "#b5542a"
COLOR_NEUTRAL = "#7f7f7f"
COLOR_LIGHT = "#d9d9d9"
COLOR_ACCENT = "#6a9e3a"
PALETTE = [COLOR_PRIMARY, COLOR_SECONDARY, COLOR_ACCENT, COLOR_NEUTRAL, COLOR_LIGHT]

import matplotlib as _mpl
_mpl.rcParams.update({
    "figure.figsize": FIGSIZE,
    "figure.dpi": DPI,
    "axes.titlesize": FONT_TITLE,
    "axes.labelsize": FONT_LABEL,
    "xtick.labelsize": FONT_TICK,
    "ytick.labelsize": FONT_TICK,
    "legend.fontsize": FONT_LEGEND,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
})


## Linkage Statistics: Individual-to-Polity Matching

This notebook shows how individuals were matched to Cliopatria polities.
Linkages are broken down by method (polygon vs. URL) and origin (nationality, birthplace, deathplace).
The resulting table is saved as a PNG for inclusion in the paper.

In [ ]:
import duckdb
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'

DB = DB_PATH

conn = duckdb.connect(DB, read_only=True)
# Count linkages by method (normalized to polygon / url) and origin
cur = conn.execute('''
    SELECT
        CASE
            WHEN method LIKE '%polygon%' THEN 'polygon'
            WHEN method LIKE '%url%'     THEN 'url'
            ELSE method
        END AS method_group,
        origin,
        COUNT(*) AS n
    FROM individuals_cliopatria
    GROUP BY method_group, origin
    ORDER BY method_group, origin
''')

detail_rows = [(m, o, n) for m, o, n in cur.fetchall()]

polygon_total = sum(n for m, _, n in detail_rows if m == 'polygon')
url_total     = sum(n for m, _, n in detail_rows if m == 'url')
grand_total   = sum(n for _, _, n in detail_rows)

conn.close()

# Build cell data
METHOD = {'polygon': 'Polygon merging', 'url': 'URL matching'}
ORIGIN = {
    'country_of_citizenship': 'Country of nationality',
    'birthplace':              'Place of birth',
    'deathplace':              'Place of death',
}

cell_text = []
for method, origin, count in detail_rows:
    cell_text.append([
        METHOD.get(method, method),
        ORIGIN.get(origin, origin),
        f'{count:,}',
        f'{count / grand_total * 100:.1f}%',
    ])

summary_rows = [
    ['', 'Polygon subtotal', f'{polygon_total:,}', f'{polygon_total/grand_total*100:.1f}%'],
    ['', 'URL subtotal',     f'{url_total:,}',     f'{url_total/grand_total*100:.1f}%'],
    ['', 'Total linkages',   f'{grand_total:,}',    '100.0%'],
]

all_rows = cell_text + summary_rows
n_detail = len(cell_text)
n_all = len(all_rows)
col_labels = ['Method', 'Origin', 'Count', '%']

# Render table — wider figure + wider Method/Origin columns to avoid text overlap
fig_height = 0.42 * (n_all + 1) + 0.6
fig, ax = plt.subplots(figsize=(9.0, fig_height), dpi=200)
ax.axis('off')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

table = ax.table(
    cellText=all_rows,
    colLabels=col_labels,
    cellLoc='center',
    colLoc='center',
    loc='center',
    edges='open',
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.6)

# Style header
for j in range(len(col_labels)):
    cell = table[0, j]
    cell.set_text_props(fontweight='bold', color='#222222', fontsize=12)
    cell.set_facecolor('white')
    cell.set_edgecolor('white')
    cell.get_text().set_ha('center')

# Style detail rows
for i in range(1, n_detail + 1):
    for j in range(len(col_labels)):
        cell = table[i, j]
        cell.set_facecolor('#fafafa' if i % 2 == 0 else 'white')
        cell.set_edgecolor('white')
        cell.set_text_props(fontsize=11, color='#333333')
        if j <= 1:
            cell.get_text().set_ha('left')
        else:
            cell.get_text().set_ha('right')

# Style summary rows
for i in range(n_detail + 1, n_all + 1):
    row_idx = i - n_detail - 1
    for j in range(len(col_labels)):
        cell = table[i, j]
        cell.set_edgecolor('white')
        if j <= 1:
            cell.get_text().set_ha('left')
        else:
            cell.get_text().set_ha('right')
        if row_idx <= 1:
            cell.set_facecolor('#f0f0f0')
            cell.set_text_props(fontweight='bold', fontsize=10.5, color='#555555')
        else:
            cell.set_facecolor('#e4e4e4')
            cell.set_text_props(fontweight='bold', fontsize=11, color='#222222')

# Column widths — wider Method/Origin to fit longer labels without overlap
col_widths = [0.28, 0.36, 0.22, 0.14]
for i in range(n_all + 1):
    for j, w in enumerate(col_widths):
        table[i, j].set_width(w)

# Draw horizontal rules
renderer = fig.canvas.get_renderer()
tbb = table.get_window_extent(renderer)
tbb_ax = tbb.transformed(ax.transData.inverted())

x_left = tbb_ax.x0
x_right = tbb_ax.x1
row_height = (tbb_ax.y1 - tbb_ax.y0) / (n_all + 1)

y_top = tbb_ax.y1
ax.plot([x_left, x_right], [y_top, y_top], color='#333333', linewidth=1.8,
        clip_on=False, transform=ax.transData)

y_below_header = tbb_ax.y1 - row_height
ax.plot([x_left, x_right], [y_below_header, y_below_header], color='#333333',
        linewidth=1.2, clip_on=False, transform=ax.transData)

y_above_summary = tbb_ax.y1 - (n_detail + 1) * row_height
ax.plot([x_left, x_right], [y_above_summary, y_above_summary], color='#999999',
        linewidth=0.8, clip_on=False, transform=ax.transData)

y_above_totals = tbb_ax.y1 - (n_detail + 3) * row_height
ax.plot([x_left, x_right], [y_above_totals, y_above_totals], color='#999999',
        linewidth=0.8, clip_on=False, transform=ax.transData)

y_bottom = tbb_ax.y0
ax.plot([x_left, x_right], [y_bottom, y_bottom], color='#333333', linewidth=1.8,
        clip_on=False, transform=ax.transData)

plt.subplots_adjust(left=0.02, right=0.98, top=0.95, bottom=0.05)
plt.show()


In [3]:
import pandas as pd

df_linkage = pd.DataFrame(detail_rows, columns=['Method', 'Origin', 'Count'])
df_linkage['Method'] = df_linkage['Method'].map({'polygon': 'Polygon merging', 'url': 'URL matching'})
df_linkage['Origin'] = df_linkage['Origin'].map({
    'country_of_citizenship': 'Country of nationality',
    'birthplace':              'Place of birth',
    'deathplace':              'Place of death',
})
df_linkage['%'] = (df_linkage['Count'] / df_linkage['Count'].sum() * 100).round(1)

print(f"Linkage statistics: {grand_total:,} total linkages")
df_linkage

Linkage statistics: 6,128,228 total linkages


,Method,Origin,Count,%
0,Polygon merging,Place of birth,655790,10.7
1,Polygon merging,Country of nationality,4128694,67.4
2,Polygon merging,Place of death,100076,1.6
3,URL matching,Place of birth,54610,0.9
4,URL matching,Country of nationality,1185389,19.3
5,URL matching,Place of death,3669,0.1
